# Further Nsight Compute Options

The default configuration of Nsight Compute, as introduced in the [Kernel Level Profiling](./04-kernel-level-profiling.ipynb) notebook, captures only a small subset of the metrics available.
To change this, different **sections** and **sets** are available, in addition to specifying own **metric** lists.

## Sections

The available sections can be queried with

In [ ]:
!ncu --list-sections > ../profiles/sections.txt

In [ ]:
!cat ../profiles/sections.txt

and used with the `--section` parameter

In [ ]:
!ncu -s 2 -c 1 --section=SpeedOfLight ../build/stencil-2d-omp-target-v3 double 8192 8192 2 2

## Sets

In many cases, multiple sections are required concurrently.
Sets provide an interface for the most relevant combinations.
As before, they can be queried directly from `ncu` and then used with the corresponding command line argument.

In [ ]:
!ncu --list-sets > ../profiles/sets.txt

In [ ]:
!cat ../profiles/sets.txt

In [ ]:
!ncu -s 2 -c 1 --set=roofline ../build/stencil-2d-omp-target-v3 double 8192 8192 2 2

If further information is necessary, the section definitions included in the Nsight Compute distribution can be helpful.
After locating them, they can simply be read.

In [ ]:
!ls -la /opt/nvidia/hpc_sdk/Linux_x86_64/26.1/profilers/13.1/Nsight_Compute/sections/

In [ ]:
!cat /opt/nvidia/hpc_sdk/Linux_x86_64/26.1/profilers/13.1/Nsight_Compute/sections/SpeedOfLight.section

## Metrics

In some cases, directly accessing very specific metrics can be helpful.
For instance, for doing automatic benchmarking or to keep profiling overheads low.

Additional material is available on the [metrics structure](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-structure), as well as a [metrics reference](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-reference).

In [ ]:
!ncu --list-metrics > ../profiles/metrics.txt

In [ ]:
!cat ../profiles/metrics.txt

Relevant metrics include (but are not limited to)
* `sm__warps_active.avg.pct_of_peak_sustained_active` displays the achieved occupancy.
* `dram__bytes_read` and `dram__bytes_write` correspond to the bytes read from and written to DRAM. This can be
    extended with `.sum` to obtain the total volume and `.sum.per_second` to obtain the bandwidth.
* `smsp__sass_thread_inst_executed_op_{dadd, dmul, dfma}_pred_on.sum` represents the total executed additions,
    multiplications and fused multiply-adds in double precision. The total number of FLOPs in double precision can be
    computed with `dadd + dmul + 2 * dfma`.
* `lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_{ld, st}.sum` can be used to query the L2
    cache load and store volumes.

In [ ]:
!ncu -s 2 -c 1 --metrics \
sm__warps_active.avg.pct_of_peak_sustained_active,dram__bytes_read.sum,dram__bytes_write.sum,dram__bytes_read.sum.per_second,dram__bytes_write.sum.per_second,smsp__sass_thread_inst_executed_op_dadd_pred_on.sum,smsp__sass_thread_inst_executed_op_dmul_pred_on.sum,smsp__sass_thread_inst_executed_op_dfma_pred_on.sum,smsp__sass_thread_inst_executed_op_dadd_pred_on.sum.per_second,smsp__sass_thread_inst_executed_op_dmul_pred_on.sum.per_second,smsp__sass_thread_inst_executed_op_dfma_pred_on.sum.per_second \
../build/stencil-2d-omp-target-v3 double 8192 8192 2 2

For counting metrics (e.g. bytes transferred) the structure is usually `metric.sum`.
For corresponding rates this can be extended to `metric.sum.per_second`.
This can also be recomputed in terms of percentage of theoretical peak performance with `metric.sum.pct_of_peak_sustained_elapsed`.

Available metrics can be queried with (example for A100)

In [ ]:
!ncu --query-metrics --chip ga100 > ../profiles/metrics.ga100.txt

and with additional filters (e.g. only metrics starting with a certain string)

In [ ]:
!ncu --query-metrics-mode suffix --metrics sm__inst_executed --chip ga100

A small collection of useful metrics is collected below:

## Metrics Selection

### Time


`gpu__time_duration.sum`

### Compute - Thread Level

Total number of instructions for *single precision* FMA, ADD, MUL
* `smsp__sass_thread_inst_executed_op_ffma_pred_on.sum`
* `smsp__sass_thread_inst_executed_op_fadd_pred_on.sum`
* `smsp__sass_thread_inst_executed_op_fmul_pred_on.sum`

Total number of instructions for *double precision* FMA, ADD, MUL
* `smsp__sass_thread_inst_executed_op_dfma_pred_on.sum`
* `smsp__sass_thread_inst_executed_op_dadd_pred_on.sum`
* `smsp__sass_thread_inst_executed_op_dmul_pred_on.sum`

The number of FLOPs in a given precision is 2 * FMA + ADD + MUL

Corresponding rates (instruction/second)
* `smsp__sass_thread_inst_executed_op_ffma_pred_on.sum.per_second`
* `smsp__sass_thread_inst_executed_op_fadd_pred_on.sum.per_second`
* `smsp__sass_thread_inst_executed_op_fmul_pred_on.sum.per_second`


* `smsp__sass_thread_inst_executed_op_dfma_pred_on.sum.per_second`
* `smsp__sass_thread_inst_executed_op_dadd_pred_on.sum.per_second`
* `smsp__sass_thread_inst_executed_op_dmul_pred_on.sum.per_second`

Total number of integer operations
* `sm__sass_thread_inst_executed_op_integer_pred_on.sum`

### Compute - Warp Level

Counts FMA warp level instructions
* `smsp__inst_executed_pipe_fma.sum`

### Main Memory

Total number of bytes transferred from/ to DRAM 
* `dram__bytes_read.sum`
* `dram__bytes_write.sum`

The corresponding rates/ bandwidths (bytes/second)
* `dram__bytes_read.sum.per_second`
* `dram__bytes_write.sum.per_second`

### L2 Cache

The number of bytes transferred for load and store operations
* `lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_ld.sum`
* `lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_st.sum`

as well as for atomic updates/ additions
* `lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_atom.sum`
* `lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_red.sum`

### Sectors

The corresponding number of sectors
* `dram__sectors_read.sum`
* `dram__sectors_write.sum`

The average bytes used per global memory sector accessed
* `smsp__sass_average_data_bytes_per_sector_mem_global.ratio`

### Launch Statistics

* `smsp__warps_launched.sum`

### Atomics

* `smsp__inst_executed_op_global_red.sum`

## Next Step

With a better understanding of how Nsight Compute's data collection can be tuned for different use cases, we continue in the [Nsight Compute GUI](./10-nsight-compute-gui.ipynb) notebook.